# CT 타일 학습 — v4.1 nobig + **동분포(known-type) 이미지단위 split**

배포=알려진 47셀 타입 재검사 시나리오 → 셀 그룹분할이 아니라 **이미지 단위 stratified 분할**(모든 셀이 train·val 양쪽).
- baseline(plain seg head)과 **config·타일·레시피 전부 동일**, **split 방식만** 다름.
- eval = val 이미지(모델이 본 타입의 새 이미지)에 SAHI conf 스윕 → **동분포 F1**(남들 0.9x와 같은 시험지).
- held-out(0.73/0.86)을 대체하는 게 아니라 **다른 배포 시나리오의 숫자를 추가**. 둘 다 정직하게 병기.


In [1]:
!df -h /content /

Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   47G  189G  20% /
overlay         236G   47G  189G  20% /


In [ ]:
# == §0 셋업 + CT 데이터 로케이트 (Colab v4.1) ==
!pip -q install ultralytics shapely pandas pyyaml
import os, shutil, subprocess, random, zipfile
from pathlib import Path
import numpy as np, pandas as pd
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

# 환경 바뀌면 아래 ROOT/DATA만 수정
ROOT      = Path('/content/drive/MyDrive/battery_yolo')
PROJECT   = ROOT/'빅프로젝트'                                         # 읽기전용(가중치 읽기용)
DATA      = ROOT/'data/battery_v4_1_output'
if not DATA.exists():
    cands=sorted((ROOT/'data').glob('*v4*1*')); assert cands, f'{DATA} 없음 — DATA 직접지정'
    DATA=cands[0]; print('자동탐색 DATA:', DATA)
DRIVE_OUT = ROOT/'kt_out_1'; DRIVE_OUT.mkdir(parents=True, exist_ok=True)
RUNS      = DRIVE_OUT/'runs_main'; RUNS.mkdir(parents=True, exist_ok=True)   # 학습출력(세션 죽어도 resume)

# manifest: loose 파일 우선, 없으면 zip 안에서 추출
def find_meta(name):
    hits=sorted(DATA.rglob(name))
    if hits: return hits[0]
    for z in sorted(DATA.rglob('*.zip')):
        try: zf=zipfile.ZipFile(z)
        except zipfile.BadZipFile: continue
        for n in zf.namelist():
            if n.replace('\\','/').split('/')[-1]==name:
                zf.extract(n,'/content/work/v41_meta'); return Path('/content/work/v41_meta')/n
    return None
MANIFEST=find_meta('manifest.csv'); assert MANIFEST, f'manifest 못찾음: {DATA}'
print('manifest:', MANIFEST)

# 동분포 split: 셀 그룹분할이 아니라 이미지 단위 → 47셀 모두 train·val 양쪽
mani = pd.read_csv(MANIFEST, dtype=str, keep_default_na=False)
seg  = mani[(mani['modality']=='CT') & (mani['included_seg'].str.lower()=='true')].copy()
dev  = seg                       # 알려진 타입 전량을 학습풀로
VAL_FRAC = 0.15
val_idx=[]
for cid,grp in dev.groupby('battery_id'):
    k = max(1,int(round(len(grp)*VAL_FRAC))) if len(grp)>=2 else 0   # 셀당 ≥1장은 반드시 train에 남김
    if k: val_idx += list(grp.sample(n=k, random_state=42).index)
vmask = dev.index.isin(val_idx)
ct_split = {'val': dev[vmask], 'train': dev[~vmask]}
print('CT 동분포 split | 전체', len(dev), '| train', len(ct_split['train']), '| val', len(ct_split['val']), '| 셀', dev['battery_id'].nunique())

# 파일명으로 인덱싱(name→이미지, stem→labels_seg) = zip 내부구조 무관.
# dev 커버리지 부족하면 CT zip을 채워질 때까지 순차 해제.
LOCAL = Path('/content/work/data/ct'); LOCAL.mkdir(parents=True, exist_ok=True)
IMG_EXTS={'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}
def build_index():
    imgs={}; segs={}
    for f in LOCAL.rglob('*'):
        if not f.is_file(): continue
        sfx=f.suffix.lower()
        if sfx in IMG_EXTS: imgs[f.name]=f
        elif sfx=='.txt' and f.parent.name=='labels_seg': segs[f.stem]=f
    return imgs,segs
IMG_INDEX,SEG_INDEX=build_index()
def dev_cov(): return sum(1 for n in dev['output_image_name'] if n in IMG_INDEX)
if dev_cov() < len(dev)*0.95:
    zips=sorted(DATA.rglob('*CT*.zip')) or sorted(DATA.rglob('*.zip')); assert zips, f'CT zip 없음: {DATA}'
    for z in zips:
        print('해제:', z.name, '(dev 이미지 채울 때까지)')
        r=subprocess.run(['unzip','-q','-o',str(z),'-d',str(LOCAL)],capture_output=True,text=True)
        assert r.returncode<=1,(r.stderr or r.stdout)[-500:]
        IMG_INDEX,SEG_INDEX=build_index()
        if dev_cov()>=len(dev)*0.95: break
cov=dev_cov()
print('이미지 인덱스:',len(IMG_INDEX),'| labels_seg 인덱스:',len(SEG_INDEX),'| dev 커버리지:',f'{cov}/{len(dev)}')
assert cov>=len(dev)*0.95, f'dev 이미지 매칭 부족 {cov}/{len(dev)} — zip/구조 확인'

자동탐색 DATA: /content/drive/MyDrive/battery_yolo/data/battery_v41_output
manifest: /content/drive/MyDrive/battery_yolo/data/battery_v41_output/reports/manifest.csv
CT 동분포 split | 전체 67605 | train 57465 | val 10140 | 셀 47
이미지 인덱스: 67607 | labels_seg 인덱스: 67605 | dev 커버리지: 67605/67605


In [ ]:
# == §0.5 데이터 점검 (동분포/known-type 게이트) ==
tr_ids=set(ct_split['train']['battery_id']); va_ids=set(ct_split['val']['battery_id'])
print('CT 셀 =', len(tr_ids|va_ids), '(기대 47) | train셀', len(tr_ids), '| val셀', len(va_ids))
print('val 셀이 train에도 다 있나(동분포 핵심):', va_ids<=tr_ids, '| 양쪽 공통', len(tr_ids&va_ids))
assert (tr_ids|va_ids) and va_ids<=tr_ids, '동분포 게이트 실패: val 셀이 train에 없음(=held-out 되어버림)'
n_big=int((pd.to_numeric(seg['porosity_bbox_max_ratio'],errors='coerce').fillna(0)>=0.25).sum())
print('included_seg 대형(≥25%) 라벨:', n_big, '장 (기대 0 = v4.1 nobig)')
assert n_big==0, f'대형 오라벨 {n_big}장 잔존'
print('→ 동분포(known-type): 모든 셀 train·val 양쪽. eval=val 이미지(모델이 본 타입의 새 이미지) → 배포 대표숫자.')


CT 셀 = 47 (기대 47) | train셀 47 | val셀 47
val 셀이 train에도 다 있나(동분포 핵심): True | 양쪽 공통 47
included_seg 대형(≥25%) 라벨: 0 장 (기대 0 = v4.1 nobig)
→ 동분포(known-type): 모든 셀 train·val 양쪽. eval=val 이미지(모델이 본 타입의 새 이미지) → 배포 대표숫자.


In [ ]:
# == §1 config (동분포: baseline과 동일 레시피, split만 다름) ==
SEED=42; random.seed(SEED)
CFG=dict(amp=True, lr0=0.0005, epochs=18, patience=15,
         hsv_h=0.0, hsv_s=0.0, hsv_v=0.1, degrees=10, flipud=0.5, translate=0.2,
         scale=0.1, copy_paste=0.0, mask_ratio=2, overlap_mask=False)
SAVE_PERIOD=1
DROP_BIG_LABELS=True; BIG_AREA_THR=0.25
RUN_NAME='train_ct_tiled_v41_samedist'
TILED=Path('/content/work/datasets/ct_tiled_v41_samedist_bg08')    # split 다르면 타일 새로 빌드(다른 split 타일과 섞이면 안 됨)
BACKUP=DRIVE_OUT/'ct_tiled_v41_samedist_best_backup.pt'
print(f'동분포 | DROP_BIG_LABELS={DROP_BIG_LABELS} | RUN_NAME={RUN_NAME}')
print('CFG:',CFG); print('tiled:',TILED,'| backup:',BACKUP)


동분포 | DROP_BIG_LABELS=True | RUN_NAME=train_ct_tiled_v41_samedist
CFG: {'amp': True, 'lr0': 0.0005, 'epochs': 18, 'patience': 15, 'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.1, 'degrees': 10, 'flipud': 0.5, 'translate': 0.2, 'scale': 0.1, 'copy_paste': 0.0, 'mask_ratio': 2, 'overlap_mask': False}
tiled: /content/work/datasets/ct_tiled_v41_samedist_bg08 | backup: /content/drive/MyDrive/battery_yolo/kt_out_1/ct_tiled_v41_samedist_best_backup.pt


In [ ]:
# == §2 타일 데이터셋 빌드 (폭 네이티브 보존 + shapely 클립) ==
from PIL import Image
from shapely.geometry import Polygon, box as shbox
Image.MAX_IMAGE_PIXELS=None

def bbox_area(pts):
    xs=[p[0] for p in pts]; ys=[p[1] for p in pts]
    return (max(xs)-min(xs))*(max(ys)-min(ys))
def read_poly(lp):
    out=[]
    if lp is not None and lp.exists():
        for ln in lp.read_text().splitlines():
            v=ln.split()
            if len(v)>=7:
                cls=int(float(v[0])); xy=list(map(float,v[1:]))
                out.append((cls,list(zip(xy[0::2],xy[1::2]))))
    return out
def tile_starts(L,T,step):
    if L<=T: return [0]
    xs=list(range(0,L-T+1,step))
    if xs[-1]!=L-T: xs.append(L-T)
    return xs

Ws = pd.to_numeric(dev['roi_w'],errors='coerce').dropna()
Wmax=int(Ws.max())
print(f'dev {len(dev)}장 | roi 폭 min{int(Ws.min())} p50{int(Ws.median())} p95{int(Ws.quantile(.95))} max{Wmax}')
OVERLAP,BG_KEEP,N_BG_TILES = 0.2,0.08,1   # 배경 타일 비율. 올리면 precision↑ 학습시간↑
TILE=int(np.ceil(max(1280,Wmax)/32)*32)
assert TILE<=2048, f'폭 max {Wmax} → TILE {TILE} 과대. 이상치 확인'
print(f'-> TILE={TILE} (가로 1칸/세로 스트립, 폭 네이티브) | DROP_BIG_LABELS={DROP_BIG_LABELS}')

def clip_to_tile(polys,x0,y0,tw,th,W,H):
    tb=shbox(x0,y0,x0+tw,y0+th); out=[]
    for cls,pts in polys:
        ap=[(px*W,py*H) for px,py in pts]
        if len(ap)<3: continue
        g=Polygon(ap)
        if not g.is_valid: g=g.buffer(0)
        if g.is_empty: continue
        it=g.intersection(tb)
        if it.is_empty: continue
        for gg in (it.geoms if it.geom_type.startswith('Multi') else [it]):
            if gg.geom_type!='Polygon' or gg.area<4: continue
            ex=list(gg.exterior.coords)[:-1]
            out.append((cls,[(min(max((x-x0)/tw,0),1),min(max((y-y0)/th,0),1)) for x,y in ex]))
    return out
def slice_one(img_path,polys,split,tag):
    im=Image.open(img_path).convert('RGB'); W,H=im.size
    step=int(TILE*(1-OVERLAP))
    xs=tile_starts(W,TILE,step); ys=tile_starts(H,TILE,step)
    defect,empty=[],[]
    for y0 in ys:
        for x0 in xs:
            tw=min(TILE,W-x0); th=min(TILE,H-y0)
            kp=clip_to_tile(polys,x0,y0,tw,th,W,H)
            (defect if kp else empty).append((x0,y0,tw,th,kp))
    chosen=defect+[t for t in empty if random.random()<BG_KEEP] if defect else (random.sample(empty,min(N_BG_TILES,len(empty))) if empty else [])
    for x0,y0,tw,th,kp in chosen:
        fn=f'{tag}_{x0}_{y0}'
        im.crop((x0,y0,x0+tw,y0+th)).save(TILED/'images'/split/f'{fn}.jpg',quality=95)
        with open(TILED/'labels'/split/f'{fn}.txt','w') as f:
            for cls,npts in kp:
                f.write(str(cls)+' '+' '.join(f'{x:.6f} {y:.6f}' for x,y in npts)+'\n')
    return len(chosen),len(defect)

done=TILED/'.done'
if done.exists():
    print('스킵(이미 빌드됨):',TILED,'- 재빌드하려면 .done 삭제')
else:
    for split,rows in ct_split.items():
        (TILED/'images'/split).mkdir(parents=True,exist_ok=True)
        (TILED/'labels'/split).mkdir(parents=True,exist_ok=True)
        tot=dtot=miss=excl=0
        for img,stem in zip(rows['output_image_name'],rows['output_label_stem']):
            si=IMG_INDEX.get(img)
            if si is None: miss+=1; continue
            polys=read_poly(SEG_INDEX.get(stem))
            if DROP_BIG_LABELS and any(bbox_area(p[1])>=BIG_AREA_THR for p in polys):
                excl+=1; continue   # 오라벨 포함 이미지 통째 제외(무결성)
            n,d=slice_one(si,polys,split,stem)
            tot+=n; dtot+=d
        print(f'  {split}: {tot} tiles (결함타일 {dtot}, 오라벨이미지제외 {excl}, 원본누락 {miss})')
    done.write_text('ok')

import yaml
ct_tiled_yaml=TILED/'data.yaml'
ct_tiled_yaml.write_text(yaml.safe_dump({'path':str(TILED),'train':'images/train','val':'images/val','names':['porosity']},allow_unicode=True,sort_keys=False),encoding='utf-8')
print('tiled yaml:',ct_tiled_yaml,'| TILE=',TILE)

dev 67605장 | roi 폭 min361 p50437 p951161 max1231
-> TILE=1280 (가로 1칸/세로 스트립, 폭 네이티브) | DROP_BIG_LABELS=True
  val: 12180 tiles (결함타일 3946, 오라벨이미지제외 0, 원본누락 0)
  train: 68079 tiles (결함타일 21191, 오라벨이미지제외 0, 원본누락 0)
tiled yaml: /content/work/datasets/ct_tiled_v41_samedist_bg08/data.yaml | TILE= 1280


In [ ]:
# == §4 동분포 eval: val 이미지에 SAHI conf 스윕 → 동분포 F1 ==
# 단독 실행 OK: §0·§1만 먼저(재학습 불필요).
import logging, warnings, gc, time, random
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import torch

# sahi 로그 소음 차단(이미지당 수십 줄 찍음)
for _n in list(logging.root.manager.loggerDict):
    if _n.split('.')[0] in ('sahi', 'ultralytics'):
        _lg = logging.getLogger(_n); _lg.setLevel(logging.ERROR); _lg.propagate = False
logging.getLogger('sahi').setLevel(logging.ERROR)
logging.getLogger('sahi').propagate = False
warnings.filterwarnings('ignore')

SLICE, OV, BASE_CONF, PP, IOU_HIT = 1280, 0.2, 0.02, 0.5, 0.1   # 학습 TILE(1280) 정합 = 배포 프로토콜
SWEEP = [0.02, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40]
EVAL_EPOCHS = [6, 7]   # 평가할 에폭

# 평가 장수가 실행시간을 결정(이미지당 2~10초). 서브셋은 에폭 선택용, 헤드라인은 전량.
EVAL_N = None
BALANCED = True    # True=양/음 절반씩(빠름·에폭 선택용). False=자연분포 유지
SEED = 42

# 가중치가 여러 폴더에 흩어져 있어 순서대로 탐색
MYDRIVE = Path('/content/drive/MyDrive')
WROOTS = [DRIVE_OUT/'runs_main'/RUN_NAME/'weights',
          MYDRIVE/'kt_out'/'runs_main'/RUN_NAME/'weights',
          ROOT/'runs_main'/RUN_NAME/'weights',
          PROJECT/'runs_main'/RUN_NAME/'weights']
for _d in sorted(MYDRIVE.glob('kt_out*')):
    _w = _d/'runs_main'/RUN_NAME/'weights'
    if _w not in WROOTS: WROOTS.append(_w)

print('■ 가중치 탐색 경로별 보유 에폭:')
for _r in WROOTS:
    if _r.exists():
        _eps = sorted(int(''.join(filter(str.isdigit, p.stem))) for p in _r.glob('epoch*.pt'))
        print(f'   ✅ {_r}\n      → epoch {_eps}')
    else:
        print(f'   ✗  {_r} (없음)')

def find_epoch(e):
    nm = f'epoch{e}.pt'
    for root in WROOTS:
        if (root/nm).exists(): return root/nm
    for base in [MYDRIVE, PROJECT]:                        # 최후: 전체 탐색
        hits = [h for h in sorted(base.rglob(nm)) if RUN_NAME in str(h)]
        if hits: return hits[0]
    return None

def read_gt_norm(lp):
    b = []
    if lp is not None and lp.exists() and lp.stat().st_size:
        for ln in lp.read_text().splitlines():
            v = ln.split()
            if len(v) >= 7:
                xy = list(map(float, v[1:])); xs = xy[0::2]; ys = xy[1::2]
                b.append((min(xs), min(ys), max(xs), max(ys)))
    return b
def iou(a, b):
    ix0, iy0 = max(a[0], b[0]), max(a[1], b[1]); ix1, iy1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix1-ix0) * max(0, iy1-iy0)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/ua if ua > 0 else 0.0

# val 목록 구성 + 서브샘플
ALL = []
for img, stem in zip(ct_split['val']['output_image_name'], ct_split['val']['output_label_stem']):
    ip = IMG_INDEX.get(img)
    if ip is None: continue
    lp = SEG_INDEX.get(stem); has = lp is not None and lp.stat().st_size > 0
    ALL.append((ip, lp, has))
assert ALL, 'val 이미지 0장'
pos = [x for x in ALL if x[2]]; neg = [x for x in ALL if not x[2]]
random.seed(SEED)
if EVAL_N is None:
    VITEMS = ALL
elif BALANCED:
    k = EVAL_N // 2
    VITEMS = random.sample(pos, min(k, len(pos))) + random.sample(neg, min(k, len(neg)))
    random.shuffle(VITEMS)
else:
    VITEMS = random.sample(ALL, min(EVAL_N, len(ALL)))
np_, nn_ = sum(x[2] for x in VITEMS), sum(not x[2] for x in VITEMS)
print(f'동분포 val 전체 {len(ALL)}장(양 {len(pos)}/음 {len(neg)}) → 평가 {len(VITEMS)}장 (양 {np_}/음 {nn_})')
if EVAL_N is not None and BALANCED:
    print('  ⚠️ balanced 서브셋은 precision이 낙관적으로 나옴(base-rate 효과, 0725 §6 확인).')
    print('     에폭 선택용으로만 쓰고, 최종 헤드라인은 peak 에폭에 EVAL_N=None으로 재실행할 것.')

def prf(tp, fp, fn):
    P = tp/(tp+fp) if tp+fp else 0.; R = tp/(tp+fn) if tp+fn else 0.
    return P, R, (2*P*R/(P+R) if P+R else 0.)

def sweep_weight(wp):
    m = AutoDetectionModel.from_pretrained(model_type='ultralytics', model_path=str(wp),
                                           confidence_threshold=BASE_CONF, device='cuda:0')
    RES = []; t0 = time.time()
    for i, (ip, lp, g) in enumerate(VITEMS):
        r = get_sliced_prediction(str(ip), m, slice_height=SLICE, slice_width=SLICE,
                                  overlap_height_ratio=OV, overlap_width_ratio=OV,
                                  postprocess_match_threshold=PP, verbose=0)
        W, H = r.image_width, r.image_height
        preds = [(o.bbox.to_xyxy(), o.score.value) for o in r.object_prediction_list]
        gts = [(x0*W, y0*H, x1*W, y1*H) for x0, y0, x1, y1 in read_gt_norm(lp)]
        RES.append((preds, gts, g))
        if (i+1) % 25 == 0 or i+1 == len(VITEMS):
            el = time.time()-t0; eta = el/(i+1)*(len(VITEMS)-i-1)
            print(f'    {i+1}/{len(VITEMS)}  경과 {el/60:.1f}분  남은 ~{eta/60:.1f}분', flush=True)
    del m; gc.collect(); torch.cuda.empty_cache()
    rows = []
    for conf in SWEEP:
        itp = ifp = ifn = ltp = lfn = lfp = 0
        for preds, gts, g in RES:
            pk = [(b, s) for b, s in preds if s >= conf]; fired = len(pk) > 0
            itp += fired and g; ifp += fired and not g; ifn += (not fired) and g
            if g:
                loc = any(iou(b, gb) > IOU_HIT for b, _ in pk for gb in gts); ltp += loc; lfn += not loc
            elif fired: lfp += 1
        iP, iR, iF = prf(itp, ifp, ifn); lF = prf(ltp, lfp, lfn)[2]
        rows.append(dict(conf=conf, iP=iP, iR=iR, iF=iF, lF=lF,
                         loc=(ltp/(ltp+lfn) if ltp+lfn else 0), ifp=ifp, ifn=ifn))
    return rows

best_overall = None; lines = []
for e in EVAL_EPOCHS:
    wp = find_epoch(e)
    if wp is None: print('스킵(없음): epoch', e); continue
    print(f'\n[epoch{e}] ← {wp}', flush=True)
    rows = sweep_weight(wp)
    print('='*72)
    print(f'[epoch{e}] 동분포 val {len(VITEMS)}장 @ slice{SLICE}/ov{OV}')
    print(f'{"conf":>5} | {"img_P":>6} {"img_R":>6} {"img_F1":>6} | {"loc_F1":>6} {"loc%":>6} | {"FP":>5} {"FN":>4}')
    for s in rows:
        print(f'{s["conf"]:>5.2f} | {s["iP"]:>6.3f} {s["iR"]:>6.3f} {s["iF"]:>6.3f} | '
              f'{s["lF"]:>6.3f} {s["loc"]:>6.1%} | {s["ifp"]:>5} {s["ifn"]:>4}')
    b = max(rows, key=lambda s: s['iF'])
    print(f'  → F1-max: conf {b["conf"]:.2f} P{b["iP"]:.3f}/R{b["iR"]:.3f}/F1 {b["iF"]:.3f} | '
          f'loc-F1 {b["lF"]:.3f}/loc {b["loc"]:.1%}')
    lines.append(f'epoch{e}: F1 {b["iF"]:.3f}@conf{b["conf"]:.2f} (P{b["iP"]:.3f}/R{b["iR"]:.3f}) loc-F1 {b["lF"]:.3f}')
    if best_overall is None or b['iF'] > best_overall[1]['iF']: best_overall = (f'epoch{e}', b)

print('\n' + '#'*72)
if best_overall:
    n, b = best_overall
    print(f'🏆 동분포 최고 = [{n}] conf {b["conf"]:.2f} → img F1 {b["iF"]:.3f} '
          f'(P{b["iP"]:.3f}/R{b["iR"]:.3f}) | loc-F1 {b["lF"]:.3f}')
    print(f'   평가셋 {len(VITEMS)}장' + (' (balanced 서브셋 = precision 낙관)' if EVAL_N and BALANCED else ' (전량)'))
    print(f'   ▶ 다음: EVAL_EPOCHS=[{n.replace("epoch","")}], EVAL_N=None 으로 재실행 = 헤드라인 확정')
    txt = '\n'.join(lines) + f'\n최고: {n} F1 {b["iF"]:.3f} (평가 {len(VITEMS)}장)'
    for _d in [DRIVE_OUT, ROOT, Path('/content')]:      # kt_out_1 읽기전용 대비
        try:
            _p = _d/'ct_samedist_eval.txt'; _p.write_text(txt, encoding='utf-8')
            print('   요약 →', _p); break
        except Exception:
            continue
    else:
        print('   (요약 저장 실패 — 위 표를 복사해 두세요)')


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# == §5 셀ID별 train/val 분포 확인 ==
# 전제: §0 실행(ct_split 존재). 모든 셀이 train·val 양쪽에 있어야 동분포.
import pandas as pd
tr = ct_split['train'].groupby('battery_id').size()
va = ct_split['val'].groupby('battery_id').size()
dist = pd.DataFrame({'train': tr, 'val': va}).fillna(0).astype(int)
dist.index = dist.index.astype(str)
dist = dist.loc[sorted(dist.index, key=lambda x: int(x) if x.isdigit() else 10**9)]
dist['total'] = dist['train'] + dist['val']
dist['val%']  = (dist['val'] / dist['total'] * 100).round(1)
print(dist.to_string())
print('-'*52)
both = ((dist['train'] > 0) & (dist['val'] > 0)).all()
print(f'셀 수: {len(dist)} (기대 47)')
print(f'val 0장(=1장뿐이라 train전용) 셀: {list(dist.index[dist["val"]==0]) or "없음"}')
print(f'train 0장 셀: {list(dist.index[dist["train"]==0]) or "없음"}')
print(f'✅ 모든 셀이 train·val 양쪽에 있나(동분포 핵심): {bool(both)}')
print(f'합계  train {int(dist["train"].sum())} / val {int(dist["val"].sum())}'
      f'  (val 비율 {dist["val"].sum()/dist["total"].sum():.1%})')

In [ ]:
# == §6 챔피언 가중치 고정 백업 ==
# 전제: §0·§1 실행. 흩어진 가중치를 한 파일로 고정.
import shutil, hashlib, json as _json
from pathlib import Path

EPOCH = 6
TAG   = f'ct_samedist_CHAMPION_ep{EPOCH}'

# 1) 소스 찾기
MYDRIVE = Path('/content/drive/MyDrive')
ROOTS = [DRIVE_OUT/'runs_main'/RUN_NAME/'weights',
         MYDRIVE/'kt_out'/'runs_main'/RUN_NAME/'weights',
         ROOT/'runs_main'/RUN_NAME/'weights',
         PROJECT/'runs_main'/RUN_NAME/'weights']
SRC = next((r/f'epoch{EPOCH}.pt' for r in ROOTS if (r/f'epoch{EPOCH}.pt').exists()), None)
if SRC is None:
    hits = [h for h in sorted(MYDRIVE.rglob(f'epoch{EPOCH}.pt')) if RUN_NAME in str(h)]
    SRC = hits[0] if hits else None
assert SRC, f'★epoch{EPOCH}.pt 못찾음 — ROOTS 확인'
print('소스:', SRC, f'({SRC.stat().st_size/1e6:.1f} MB)')

# 2) 쓰기 가능한 목적지(읽기전용 폴더 대비)
def writable(*cands):
    for c in cands:
        try:
            c.mkdir(parents=True, exist_ok=True)
            t = c/'.wtest'; t.write_text('ok'); t.unlink(); return c
        except Exception: continue
    return None
DST_DIR = writable(MYDRIVE/'kt_out'/'champions', ROOT/'champions',
                   DRIVE_OUT/'champions', MYDRIVE/'kt_out_mine'/'champions')
assert DST_DIR, '★쓰기 가능한 백업 폴더 없음'
DST = DST_DIR/f'{TAG}.pt'

# 3) 복사 + 해시 검증(조용히 깨진 복사 방지)
def sha(p, buf=1 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while (b := f.read(buf)): h.update(b)
    return h.hexdigest()

if not DST.exists() or DST.stat().st_size != SRC.stat().st_size:
    shutil.copy2(SRC, DST)
s_src, s_dst = sha(SRC), sha(DST)
assert s_src == s_dst, f'★복사본 해시 불일치! {s_src[:12]} vs {s_dst[:12]} — 다시 실행'
print(f'✅ 백업: {DST}  sha256 {s_dst[:16]}…  검증 통과')

# 4) 메타 사이드카(운영점 숫자를 가중치 옆에 보관)
meta = {
    'champion': f'samedist plain yolo11m-seg epoch{EPOCH}',
    'run': RUN_NAME,
    'split': '동분포(known-type) 이미지단위 stratified — 47셀 전부 train·val 양쪽, 셀당 15% val, seed42',
    'data_policy': 'v4.1 nobig (대형 ≥25% 라벨 이미지 제외)',
    'inference': 'SAHI sliced slice=1024 / overlap=0.4',
    'eval_set': 'samedist val 전량 10,140장 (양 2,083 / 음 8,057)',
    'operating_points': {
        'report_f1max': {'conf': 0.15, 'P': 0.855, 'R': 0.905, 'F1': 0.879,
                         'loc_F1': 0.842, 'loc_pct': 0.839, 'FP': 320, 'FN': 197},
        'gate_high_recall': {'conf': 0.05, 'P': 0.722, 'R': 0.969, 'F1': 0.827, 'FP': 777},
    },
    'balanced_400_reference': {'conf': 0.05, 'F1': 0.925, 'note': 'precision 낙관 — 헤드라인 아님'},
    'runner_up': {'epoch': 7, 'F1': 0.862, 'loc_F1': 0.815, 'note': '고recall형(@0.02 R0.994), F1·loc 모두 열세'},
    'fp_rate_on_normal': round(320/8057, 4),
    'caveat': ('peak 에폭을 이 val로 선택 → 약한 낙관(plateau 평탄해 위험 작음). '
               '최적 conf는 배포 분포에 따라 이동(balanced 0.05 vs 자연분포 0.15).'),
    'headline': '알려진 타입 재검사 F1 0.879 (R 0.905, loc 84%) / 완전 새 타입 0.781로 재보정 필요',
    'src': str(SRC), 'sha256': s_dst,
}
DST.with_suffix('.json').write_text(_json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')
print('✅ 메타:', DST.with_suffix('.json'))
print(_json.dumps(meta['operating_points'], ensure_ascii=False, indent=2))
print(f"\n▶ 앞으로 이 파일만 쓰면 됨: {DST}")


소스: /content/drive/MyDrive/kt_out/runs_main/train_ct_tiled_v41_samedist/weights/epoch6.pt (179.6 MB)
✅ 백업: /content/drive/MyDrive/kt_out/champions/ct_samedist_CHAMPION_ep6.pt  sha256 2df71ae131ed93f0…  검증 통과
✅ 메타: /content/drive/MyDrive/kt_out/champions/ct_samedist_CHAMPION_ep6.json
{
  "report_f1max": {
    "conf": 0.15,
    "P": 0.855,
    "R": 0.905,
    "F1": 0.879,
    "loc_F1": 0.842,
    "loc_pct": 0.839,
    "FP": 320,
    "FN": 197
  },
  "gate_high_recall": {
    "conf": 0.05,
    "P": 0.722,
    "R": 0.969,
    "F1": 0.827,
    "FP": 777
  }
}

▶ 앞으로 이 파일만 쓰면 됨: /content/drive/MyDrive/kt_out/champions/ct_samedist_CHAMPION_ep6.pt
